<a href="https://colab.research.google.com/github/zjuiEMLab/rshub/blob/main/demo/Soil-demo-NMM3D-VIE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Active Soil demo

In [1]:
import datetime
import copy
!pip install rshub -q

In [ ]:
# Define user token
# Register your account to get a token https://rshub.zju.edu.cn/Registration
token = 'ENTER YOUR TOKEN HERE' # Register an account to get a token
# Change your task or project name every time you run a new job.
project_name = 'Demo'
task_name = 'NMM3D_VIE_DDA_demo'

### Step 1: Define Scenario flag

In [30]:
# ============== CHANGE YOUR INPUT PARAMETERS HERE ==============
# ====== Parameters not define will be set to default values ======

# 'soil: Bare soil
# 'snow: Snow
# 'veg: Vegetation covered soil
scenario_flag = 'soil'

### Step 2: Define observation description

In [31]:
# 1) Observation mode
# 'sigma': Active (Backscatter)
# 'tb': Passive (Brightness temperature)
output_var = 'sigma' # for soil model, both active and passive results will be outputed; Use this flag to retrieve results

# 2) Observation characteristics
fGHz = 1.26

### Step 3: Define Algorithm flag

In [32]:
# 1: VIE NMM3D
algorithm = 'vie'

### Step 4: Describe your scenario

In [ ]:
# Observation setup
fGHz = 1.26  # Microwave frequency [GHz]
angle = 30   # Incidence angle [deg]

# Simulation-domain geometry
Lx = 1.6     # Domain length in x direction [m]; ~ 7-9*wavelength 
Ly = 1.6     # Domain length in y direction [m]; ~ 7-9*wavelength 
Lz = 0.04    # Domain depth in z direction [m];~ top soil depth + 2-3*RMSHeight
xr = -0.8    # Left x boundary position [m]
yr = -0.8    # Back y boundary position [m]
zr = 0.0     # Bottom z boundary position [m]
d = 0.004    # Spatial discretization resolution [m]; suggest to be <0.02 * wavelength

# Ground dielectric and temperature
epsr_g_re = 5.98   # Real part of ground relative permittivity [-]
epsr_g_im = 0.54   # Imaginary part of ground relative permittivity [-]
Tg = 273.15        # Ground temperature [K]


# Soil profile: layers must be listed from bottom to top.
# soilType: 1 = Gaussian roughness correlation, 2 = exponential roughness correlation.
# RMSHeight: layer RMS roughness height [m]
# CLx, CLy: correlation lengths in x and y [m]
# layerZaxis: layer elevation/depth parameter [m]
# epsr_re, epsr_im: complex relative permittivity parts [-]
# Tb: layer physical temperature [K]
soil_layers_bottom_to_top = [
      {
        "soilType": 2,
        "RMSHeight": 0.0,
        "CLx": 0.0,
        "CLy": 0.0,
        "layerZaxis": 0.02,
        "epsr_re": 5.2648,
        "epsr_im": 0.453,
        "Tb": 273.15
      },
      {
        "soilType": 2,
        "RMSHeight": 0.005,
        "CLx": 0.1,
        "CLy": 0.1,
        "layerZaxis": 0.05,
        "epsr_re": 5.2304,
        "epsr_im": 0.4488,
        "Tb": 273.15
      }
]


In [ ]:
data = {
    'scenario_flag': scenario_flag,
    'output_var': output_var,
    'fGHz': fGHz,
    'algorithm': algorithm,
    'microwave': {
        'fGHz': fGHz,
        'angle': angle,
    },
    'geometry': {
        'Lx': Lx,
        'Ly': Ly,
        'Lz': Lz,
        'xr': xr,
        'yr': yr,
        'zr': zr,
        'd': d,
    },
    'soil': {
        'layers_bottom_to_top': soil_layers_bottom_to_top,
    },
    'dielectric': {
        'epsr_g_re': epsr_g_re,
        'epsr_g_im': epsr_g_im,
    },
    'temperature': {
        'Ts': Ts,
        'Tg': Tg,
    },
    'project_name': project_name,
    'task_name': task_name,
    'token': token,
    'force_update_flag': 1,  # Force replace existing task.
}

data

## Run models

In [ ]:
from rshub import submit_jobs
result=submit_jobs.run(data)

In [36]:
print(result['result'])

Job submitted!


# Check Job Status (The demo will take over three hours)

In [ ]:
from rshub import submit_jobs
result=submit_jobs.check_completion(token, project_name, task_name)
print(result)

{'task_status': 'completed', 'project_name': 'Demo', 'task_name': 'NMM3D_VIE_DDA_demo'}


In [18]:
from rshub import load_file

data = load_file(token, project_name, task_name,scenario_flag=scenario_flag,algorithm=algorithm,output_var=output_var)
message = data.load_error_message()


message: Jobs completed succesfully



# Post Process

## Active

In [21]:
# load mat file with project id, frequencies,variables to load
output_var = 'sigma'
data = load_file(token, project_name, task_name,scenario_flag=scenario_flag,algorithm=algorithm,output_var=output_var)

data_active = data.load_outputs(fGHz=fGHz, inc_ang=30) #

['Passive_fGHz1.26_ob_angle30.h5', 'Active_fGHz1.26_ob_angle30.h5']
File size: 0.70 MB
File is small (<= 50 MB), loading into memory...
Loading small file directly into memory...
Sucessfully loaded


In [24]:
data_active.keys()

dict_keys(['backscatter', 'backscatter_angle', 'g_coh', 'g_inc', 'g_tot', 'g_tot_inxozHH', 'g_tot_inxozHV', 'g_tot_inxozVH', 'g_tot_inxozVV', 'info_json', 'phis', 'theta_xoz', 'thetas'])

In [27]:
# Backscatter in order of VV, HV, VH, HH
backscatter = data_active['backscatter']

In [ ]:
VV = backscatter[0]
HV = backscatter[1]
VH = backscatter[2]
HH = backscatter[3]

print(f"VV:{VV[0][0][0][0]} dB, HV:{HV[0][0][0][0]} dB, VH:{VH[0][0][0][0]} dB, HH:{HH[0][0][0][0]} dB")